In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


c:\Users\Sakshi Sinha\Downloads\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1538.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
sample_text = """Types of Automation Tools
IT Automation Tools: These tools help automate IT processes, 
including batch processing, workflow management, and system monitoring. 
They can streamline operations, reduce manual errors, and improve visibility across IT environments. 
Examples include Stonebranch, which offers real-time hybrid IT automation, and LambdaTest, which provides cloud-based 
test orchestration for web applications. """

In [4]:
#Sentence based chunking
#Split into sentences

sentences = [s.strip() for s in sample_text.split('\n') if s.strip()]

#Encode the sentences
embeddings = model.encode(sentences)

In [9]:
#Initialize the parameters
threshold = 0.7
chunks = []
current_chunk = [sentences[0]]

In [11]:
#Checks for similairy among neighbouring sentences and groups them together if they are similar enough
#Not Globally
for i in range(1, len(sentences)):
    similaritys = cosine_similarity(
        [embeddings[i]], 
        [embeddings[i-1]]
    )[0][0]

    if similaritys >=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]
#Add the last chunk
chunks.append(" ".join(current_chunk))



In [12]:
#Output the chunks
for i, chunk in enumerate(chunks):
    print(f"Current chunk is {i}: {chunk}")

Current chunk is 0: Types of Automation Tools IT Automation Tools: These tools help automate IT processes,
Current chunk is 1: including batch processing, workflow management, and system monitoring.
Current chunk is 2: They can streamline operations, reduce manual errors, and improve visibility across IT environments.
Current chunk is 3: Examples include Stonebranch, which offers real-time hybrid IT automation, and LambdaTest, which provides cloud-based
Current chunk is 4: test orchestration for web applications. IT Automation Tools: These tools help automate IT processes,
Current chunk is 5: including batch processing, workflow management, and system monitoring.
Current chunk is 6: They can streamline operations, reduce manual errors, and improve visibility across IT environments.
Current chunk is 7: Examples include Stonebranch, which offers real-time hybrid IT automation, and LambdaTest, which provides cloud-based
Current chunk is 8: test orchestration for web applications.


RAG Pipeline Modular Coding

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chat_models import init_chat_model
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [27]:
#Custom sementic chunking using cosine similairity
class ThresholdSemanticChunker:
    def __init__(self, model_name= "all-MiniLM-L6-v2", threshold= 0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold
    
    def split(self, text: str):
        sentences = [s.strip() for s in text.split('\n') if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            similarity = cosine_similarity(
                [embeddings[i]], 
                [embeddings[i-1]]
            )[0][0]

            if similarity >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]
        chunks.append(" ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self, documents):
        results = []
        for document in documents:
            for chunks in self.split(document.page_content):
                results.append(Document(page_content = chunks, metadata=document.metadata))
        return results



    

In [28]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [29]:
### Chunking
chunker=ThresholdSemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 931.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone..'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers..'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris..'),
 Document(metadata={}, page_content='France is a popular tourist destination..')]

In [30]:
print(len(chunks))

4


In [31]:
#Vector Store
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings()
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1646.05it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
retriever = vector_store.as_retriever()

[Document(id='db0cf4ff-b444-454e-bed5-718b437dfadb', metadata={}, page_content='The Eiffel Tower is located in Paris..'),
 Document(id='26ceadf6-33c3-45c2-bd69-a0ff3b650930', metadata={}, page_content='France is a popular tourist destination..'),
 Document(id='6fcde606-7004-4073-885d-754ce44cdd38', metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone..'),
 Document(id='4a154eb6-6a89-4239-985f-8c32f3013424', metadata={}, page_content='You can create chains, agents, memory, and retrievers..')]

In [41]:
#Prompt Template
from langchain_classic.prompts import PromptTemplate
chat_prompt = """You are a helpful assistant for answering questions about the provided context.
Context: {context}
Question: {question}"""
prompt_template = PromptTemplate.from_template(
    chat_prompt
)

In [38]:
#LLM
llm = init_chat_model(model="groq:llama-3.1-8b-instant",temperature=0.4)

In [42]:
#LECL RAG Chain
rag_chain = (
    RunnableMap(
        {
            "context" : lambda x: retriever.invoke(x["question"], k=2),
            "question" : lambda x: x["question"]
        }
    )
    | prompt_template
    |llm
    |StrOutputParser()
)

query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

LangChain is a framework for building applications with Large Language Models (LLMs). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


Sementic Chunker with Langchain

In [45]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_classic.document_loaders import TextLoader

## Create the semantic chunker
chunker=SemanticChunker(embeddings)
